# ☢️ SPECT/CT 기반 3D 병변 자동 분할 대시보드
임상의의 시각적 검증 및 OpenGATE 선량평가 연동을 위한 Jupyter Notebook입니다.

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from skimage.transform import iradon, radon
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm
import os, warnings

warnings.filterwarnings("ignore")
plt.style.use('default')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white'})

print(">>> [Step 1] Initializing Environment & Configuration...")

CFG = {
    'SPECT_PATH': './data/SPECT/SPECT-CT_SC001_DS.dcm',
    'EXPORT_DIR': './Experiments/Thyroid_Recon_Paper',
    'ANGLES': 60, 'ARC': 360.0,
    'FBP_FILTER': 'shepp-logan',
    'OSEM_ITER': 4,
    'MAP_ITER': 15, 'MAP_BETA': 0.02,
    'FLIP_Z': True, 'FLIP_Y': True
}

os.makedirs(CFG['EXPORT_DIR'], exist_ok=True)
ALGO_NAMES = ['FBP', 'OSEM', 'MAP']
CMAPS = ['Greys', 'Blues', 'Reds']
print("✅ 환경 설정 완료.")

In [ ]:
print(">>> [Step 2] Loading DICOM Data & Voxel Calibration...")

if not os.path.exists(CFG['SPECT_PATH']):
    raise FileNotFoundError(f"🚨 파일을 찾을 수 없습니다: {CFG['SPECT_PATH']}\n데이터 경로를 다시 확인해 주세요.")

img = sitk.ReadImage(CFG['SPECT_PATH'])
raw = sitk.GetArrayFromImage(img).astype(np.float32)

if raw.shape[0] == CFG['ANGLES']: 
    sinograms = np.transpose(raw, (1, 0, 2))
else: 
    sinograms = raw

sp = img.GetSpacing()
vox_ml = (sp[0] * sp[0] * sp[1]) / 1000.0 

meta = {
    'spacing': (sp[0], sp[0], sp[1]), 
    'shape': (sinograms.shape[0], sinograms.shape[2], sinograms.shape[2]), 
    'vox_ml': vox_ml
}

print(f"      • Raw Data Shape : {sinograms.shape}")
print(f"      • Unit Voxel Vol : {vox_ml * 1000:.4f} mm³")
print("✅ 데이터 로드 및 전처리 완료.")

In [ ]:
print(">>> [Step 3] Running 3D Image Reconstruction Engine...")

d, angles, w = sinograms.shape
theta = np.linspace(0., CFG['ARC'], angles, endpoint=False)
v_fbp, v_osem, v_map = [np.zeros((d, w, w), dtype=np.float32) for _ in range(3)]

for z in tqdm(range(d), desc="Reconstructing Slices"):
    sino = sinograms[z].T 
    if np.max(sino) < 1.0: continue
    
    v_fbp[z] = np.maximum(iradon(sino, theta=theta, filter_name=CFG['FBP_FILTER'], circle=True), 0)
    
    est = np.ones((w, w), dtype=np.float32)
    sens = iradon(np.ones_like(sino), theta=theta, filter_name=None, circle=True) + 1e-9
    for _ in range(CFG['OSEM_ITER']): 
        est *= (iradon(sino/(radon(est, theta)+1e-9), theta, filter_name=None) / sens)
    v_osem[z] = est
    
    est_m = est.copy()
    for _ in range(CFG['MAP_ITER']):
        upd = iradon(sino/(radon(est_m, theta)+1e-9), theta, filter_name=None, circle=True)
        pen = CFG['MAP_BETA'] * (est_m - gaussian_filter(est_m, sigma=0.8))
        est_m *= (upd / (sens + pen + 1e-9))
    v_map[z] = est_m

if CFG['FLIP_Z']: v_fbp, v_osem, v_map = [np.flip(v, axis=0) for v in [v_fbp, v_osem, v_map]]
if CFG['FLIP_Y']: v_fbp, v_osem, v_map = [np.flip(v, axis=1) for v in [v_fbp, v_osem, v_map]]

vols = {'FBP': v_fbp, 'OSEM': v_osem, 'MAP': v_map}
center = np.unravel_index(np.argmax(v_map), (d, w, w))
print("✅ 3D 영상 재구성 완료.")

In [ ]:
print(">>> [Step 4] Launching Interactive Clinical Dashboard...")

def get_metrics(th):
    stats = {}
    for algo in ALGO_NAMES:
        vol = vols[algo]
        v_max = np.max(vol)
        mask = (vol / (v_max + 1e-9) * 100.0 > th).astype(np.uint8)
        stats[algo] = {'vol_ml': np.sum(mask) * meta['vox_ml'], 'mask': mask, 'max': v_max}
    return stats

sz, sy, sx = meta['shape']
hz, hy, hx = center
asp_z = meta['spacing'][2] / meta['spacing'][0]

def draw_final_dashboard(z, y, x, th):
    data = get_metrics(th)
    fig = plt.figure(figsize=(22, 11))
    gs = fig.add_gridspec(2, 4, width_ratios=[0.6, 1, 1, 1])
    plt.subplots_adjust(wspace=0.15, hspace=0.25, top=0.88, bottom=0.05, right=0.95)
    
    fig.suptitle(f"Thyroid Recon Analysis | Z:{z} Y:{y} | Voxel: {meta['vox_ml']*1000:.2f} $mm^3$ | Th: {th}%", color='#003366', fontsize=24, fontweight='heavy')

    ax_sag = fig.add_subplot(gs[0, 0])
    ax_sag.imshow(np.flipud(np.max(vols['MAP'], axis=2)), cmap='gray_r', aspect=asp_z)
    ax_sag.axhline(sz - z - 1, c='#00CC00', lw=2); ax_sag.axvline(y, c='#0066CC', lw=2, ls='--')
    ax_sag.set_title("① Sagittal Scout", color='#444444', fontsize=14); ax_sag.axis('off')

    ax_cor = fig.add_subplot(gs[1, 0])
    ax_cor.imshow(np.flipud(np.max(vols['MAP'], axis=1)), cmap='gray_r', aspect=asp_z)
    ax_cor.axhline(sz - z - 1, c='#00CC00', lw=2); ax_cor.axvline(x, c='#FF3300', lw=2, ls='--')
    ax_cor.set_title("② Coronal Scout", color='#444444', fontsize=14); ax_cor.axis('off')

    for i, algo in enumerate(ALGO_NAMES):
        col = i + 1
        vol = vols[algo]
        mask = data[algo]['mask']
        vmax = data[algo]['max']
        
        ax_ax = fig.add_subplot(gs[0, col])
        im = ax_ax.imshow(vol[z], cmap=CMAPS[i], vmin=0, vmax=vmax)
        if np.sum(mask[z]) > 0: ax_ax.contour(mask[z], colors='#FFD700', linewidths=1.5)
        ax_ax.set_title(f"{algo}\n({data[algo]['vol_ml']:.2f} ml)", color='#222222', fontsize=16)
        ax_ax.axvline(x, c='#999999', ls=':', alpha=0.8); ax_ax.axhline(y, c='#999999', ls=':', alpha=0.8)
        ax_ax.set_xticks([]); ax_ax.set_yticks([])
        cax = ax_ax.inset_axes([0.94, 0.05, 0.04, 0.4])
        plt.colorbar(im, cax=cax).ax.tick_params(labelsize=8)

        ax_co = fig.add_subplot(gs[1, col])
        ax_co.imshow(np.flipud(vol[:, y, :]), cmap=CMAPS[i], aspect=asp_z, vmin=0, vmax=vmax)
        if np.sum(np.flipud(mask[:, y, :])) > 0: ax_co.contour(np.flipud(mask[:, y, :]), colors='#FFD700', linewidths=1.5)
        ax_co.set_title(f"{algo} (Coronal)", color='#666666', fontsize=12)
        ax_co.axhline(sz - z - 1, c='#999999', ls=':', alpha=0.8); ax_co.axvline(x, c='#999999', ls=':', alpha=0.8)
        ax_co.set_xticks([]); ax_co.set_yticks([])

    plt.show()

st = {'description_width': 'initial'}
sl_z = widgets.IntSlider(value=hz, min=0, max=sz-1, description='Axial Z', style=st)
sl_y = widgets.IntSlider(value=hy, min=0, max=sy-1, description='Coronal Y', style=st)
sl_x = widgets.IntSlider(value=hx, min=0, max=sx-1, description='Sagittal X', style=st)
sl_th = widgets.FloatSlider(value=35, min=5, max=90, description='Threshold %', style=st)

out_final = widgets.Output()
def update_final(z, y, x, th):
    with out_final:
        clear_output(wait=True)
        draw_final_dashboard(z, y, x, th)

widgets.interactive_output(update_final, {'z':sl_z, 'y':sl_y, 'x':sl_x, 'th':sl_th})
display(widgets.VBox([widgets.HBox([sl_z, sl_y, sl_x]), sl_th]), out_final)

In [ ]:
print(">>> [Step 5] Extracting Validation Data & OpenGATE Bridge...")

final_th = sl_th.value
final_metrics = get_metrics(final_th)
map_target_ml = final_metrics['MAP']['vol_ml']

print(f"      • Selected Threshold : {final_th}%")
print(f"      • Final Target Volume (MAP) : {map_target_ml:.2f} ml")

"""
def export_to_opengate(mask_array, spacing, output_prefix):
    import pydicom
    export_vol = (mask_array * 255).astype(np.uint8)
    sitk_img = sitk.GetImageFromArray(export_vol)
    sitk_img.SetSpacing(spacing)
    out_path = f"{CFG['EXPORT_DIR']}/{output_prefix}_GATE_Input.mhd"
    sitk.WriteImage(sitk_img, out_path)
    print(f"      ✅ OpenGATE Voxelized Source 추출 완료: {out_path}")
"""
print("✅ 모든 분석 절차가 안전하게 종료되었습니다.")